# Evaluación Formativa 1 — Análisis exploratorio
## Dataset: *Rain in Australia* (weatherAUS)

**MCDI501 - Estadística Computacional para la Toma de Decisiones**
Magíster en Ciencia de Datos e Inteligencia Artificial, Universidad Andrés Bello

**Grupo 4:** Jeremmy Díaz Alfaro · Karim Zaid Sánchez · Laura Arenas Kosin

Este notebook sigue el mismo orden del código de ejemplo del profesor (Give Me Some Credit), pero con nuestro dataset. Llega hasta la figura 5 con el siguiente orden:

1. Estadística descriptiva (ID1.2): tipos de variables, Tabla 1, calidad de datos, Figura 1 y Figura 2.
2. Estimación puntual e intervalos de confianza (ID1.3): Tabla 2 y Figura 3
3. Prueba de hipótesis (ID1.4): Figura 4 y Figura 5.
4. Síntesis

La variable objetivo es `RainTomorrow`: si al día siguiente llovió (Yes) o no (No). Semilla fija: `42`.

## 0. Configuración y carga de datos

El archivo `weatherAUS.csv` está en `data/raw/` (descargado de Kaggle). Lo cargamos con la función de `src/carga_datos.py`, que usa ruta relativa.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# buscamos la carpeta del proyecto para poder importar src
raiz = Path.cwd().resolve()
while not (raiz / "src").exists() and raiz != raiz.parent:
    raiz = raiz.parent
sys.path.append(str(raiz))

from src.carga_datos import cargar_weather

SEED = 42
np.random.seed(SEED)

# carpeta donde se guardan las figuras del informe
FIGURAS = raiz / "figuras"
FIGURAS.mkdir(exist_ok=True)

df = cargar_weather()
TARGET = "RainTomorrow"
print("Dimensiones:", df.shape)
df.head()

### Estilo de las figuras
Usamos la misma paleta del ejemplo del profesor: fondo crema, verde azulado para la serie principal, terracota para la clase minoritaria y ocre para las líneas de referencia.

In [ ]:
TEAL  = '#2e6b75'   # serie principal
TERRA = '#a35140'   # clase minoritaria / textos de referencia
OCHRE = '#c4862f'   # líneas de referencia (media, mediana)
STEM  = '#bdb3a3'   # tallos de los gráficos lollipop
BG    = '#f6f1ea'   # fondo de los ejes
GRID  = '#ddd5c6'   # líneas de grilla

plt.rcParams.update({
    'figure.dpi': 110, 'savefig.dpi': 200,
    'font.family': 'serif', 'font.size': 10,
    'axes.facecolor': BG, 'axes.edgecolor': '#8c8578',
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.titlesize': 12, 'axes.titleweight': 'normal', 'axes.titlelocation': 'left',
    'axes.grid': True, 'grid.color': GRID, 'grid.linewidth': 0.8,
    'axes.axisbelow': True,
})

### Limpieza básica

In [ ]:
# La nubosidad 9 no es válida en octas (0 a 8), se marca como faltante
df_lim = df.copy()
for c in ['Cloud9am', 'Cloud3pm']:
    print(f"{c}: {(df_lim[c] > 8).sum()} valor(es) mayor(es) a 8 pasan a NaN")
    df_lim.loc[df_lim[c] > 8, c] = np.nan

# Los análisis que usan el objetivo trabajan solo con las filas que tienen RainTomorrow
df_obj = df_lim.dropna(subset=[TARGET]).copy()
df_obj['y'] = (df_obj[TARGET] == 'Yes').astype(int)
print(f"Filas totales: {len(df_lim):,} | con objetivo: {len(df_obj):,} | sin objetivo: {len(df_lim) - len(df_obj):,}")

## 1. Estadística descriptiva (ID1.2)

### 1.1 Tipos de variables y medidas resumen
El dataset tiene 23 columnas: una fecha, la ciudad, cuatro variables de dirección del viento (categóricas), dos variables Yes/No (`RainToday` y el objetivo `RainTomorrow`) y 16 variables numéricas medidas en la estación (temperatura, lluvia, viento, humedad, presión y nubosidad).

In [ ]:
tipos = pd.Series({
    'Date': 'Fecha',
    'Location': 'Categórica nominal (49 ciudades)',
    'MinTemp': 'Continua (°C)',
    'MaxTemp': 'Continua (°C)',
    'Rainfall': 'Continua (mm)',
    'Evaporation': 'Continua (mm)',
    'Sunshine': 'Continua (horas de sol)',
    'WindGustDir': 'Categórica nominal (16 direcciones)',
    'WindGustSpeed': 'Continua (km/h)',
    'WindDir9am': 'Categórica nominal (16 direcciones)',
    'WindDir3pm': 'Categórica nominal (16 direcciones)',
    'WindSpeed9am': 'Continua (km/h)',
    'WindSpeed3pm': 'Continua (km/h)',
    'Humidity9am': 'Continua (%)',
    'Humidity3pm': 'Continua (%)',
    'Pressure9am': 'Continua (hPa)',
    'Pressure3pm': 'Continua (hPa)',
    'Cloud9am': 'Discreta ordinal (octas, 0 a 8)',
    'Cloud3pm': 'Discreta ordinal (octas, 0 a 8)',
    'Temp9am': 'Continua (°C)',
    'Temp3pm': 'Continua (°C)',
    'RainToday': 'Binaria (Yes/No)',
    'RainTomorrow': 'Binaria (objetivo)',
}, name='Tipo')
tipos.to_frame()

**Tabla 1.** Media, mediana, desviación estándar, mínimo y máximo de las variables numéricas.

In [ ]:
numericas = df_lim.select_dtypes(include='number').columns
desc = df_lim[numericas].describe().T[['count', 'mean', '50%', 'std', 'min', 'max']]
desc['skew'] = df_lim[numericas].skew()   # asimetría (0 = simétrica)
desc['kurt'] = df_lim[numericas].kurt()   # curtosis en exceso (0 = como la normal)
desc.columns = ['n', 'Media', 'Mediana', 'Desv. est.', 'Mínimo', 'Máximo', 'Asimetría', 'Curtosis']
desc.round(2)

Para las variables categóricas no tiene sentido calcular media, así que mostramos la cantidad de categorías, la más frecuente (moda) y cuántas veces aparece.

In [ ]:
categoricas = ['Location', 'WindGustDir', 'WindDir9am', 'WindDir3pm', 'RainToday', 'RainTomorrow']
resumen_cat = df[categoricas].describe().T[['count', 'unique', 'top', 'freq']]
resumen_cat.columns = ['n (sin nulos)', 'N.º categorías', 'Moda', 'Frecuencia moda']
resumen_cat

### 1.2 Calidad de datos
Revisamos el balance del objetivo, los datos faltantes, los duplicados y si hay valores imposibles.

In [ ]:
n = len(df)
y = df[TARGET].dropna()
print(f"Días con lluvia al día siguiente : {(y == 'Yes').mean()*100:.1f}%  (sobre {len(y):,} registros con dato)")
print(f"RainTomorrow sin dato            : {df[TARGET].isna().sum():,} ({df[TARGET].isna().mean()*100:.1f}%)")
print(f"Filas duplicadas                 : {df.duplicated().sum():,}")
print(f"Repetidos (misma fecha y ciudad) : {df.duplicated(['Date', 'Location']).sum():,}")

# la nubosidad se mide en octas (0 a 8), así que 9 no debería existir
print(f"Cloud9am = 9                     : {(df['Cloud9am'] == 9).sum()}")
print(f"Cloud3pm = 9                     : {(df['Cloud3pm'] == 9).sum()}")

# valores máximos que llaman la atención
print(f"Lluvia máxima en un día          : {df['Rainfall'].max():.0f} mm")
print(f"Días con algo de lluvia (>0 mm)  : {(df['Rainfall'] > 0).mean()*100:.1f}%")

# ciudades que nunca registran horas de sol
sin_sol = df.groupby('Location')['Sunshine'].apply(lambda s: s.isna().all())
print(f"Ciudades sin ningún dato de Sunshine: {sin_sol.sum()} de {df['Location'].nunique()}")

In [ ]:
# chequeo: RainToday es 'Yes' cuando llovió más de 1 mm ese día
ok = df.dropna(subset=['Rainfall', 'RainToday'])
coincide = ((ok['Rainfall'] > 1) == (ok['RainToday'] == 'Yes')).mean()
print(f"RainToday coincide con Rainfall > 1 mm en el {coincide*100:.1f}% de los registros")
assert coincide == 1.0

### Figura 1 - Variable objetivo y datos faltantes

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 5.5), gridspec_kw={'width_ratios': [1, 1.2]})

# (a) distribución de la variable objetivo (barras horizontales)
vc = df[TARGET].value_counts(dropna=False)
vc.index = ['No llueve (No)' if v == 'No' else 'Llueve (Yes)' if v == 'Yes' else 'Sin dato' for v in vc.index]
vc = vc.reindex(['No llueve (No)', 'Llueve (Yes)', 'Sin dato'])
ax[0].barh(vc.index, vc.values, color=[TEAL, TERRA, STEM], height=0.5)
ax[0].invert_yaxis()
for i, v in enumerate(vc.values):
    ax[0].text(v + 2000, i, f'{v:,}  ({v/n*100:.1f}%)', va='center', fontsize=9)
ax[0].set_xlim(0, vc.max() * 1.3)
ax[0].set_xlabel('Frecuencia')
ax[0].set_title('(a) Distribución de RainTomorrow')
ax[0].grid(axis='y', visible=False)

# (b) porcentaje de faltantes por variable (lollipop horizontal)
miss = (df.isna().mean() * 100).sort_values(ascending=False)
miss = miss[miss > 0]
pos = np.arange(len(miss))
ax[1].hlines(pos, 0, miss.values, color=STEM, lw=2.5)
ax[1].scatter(miss.values, pos, s=60, color=OCHRE, edgecolor='#333333', linewidth=0.6, zorder=3)
for i, v in enumerate(miss.values):
    ax[1].text(v + 0.8, i, f'{v:.1f}%', va='center', fontsize=8)
ax[1].set_yticks(pos)
ax[1].set_yticklabels(miss.index, fontsize=8)
ax[1].invert_yaxis()
ax[1].set_xlim(0, miss.max() * 1.2)
ax[1].set_xlabel('% faltante')
ax[1].set_title('(b) Datos faltantes por variable')
ax[1].grid(axis='y', visible=False)

plt.tight_layout()
plt.savefig(FIGURAS / 'fig_panorama.png', bbox_inches='tight')
plt.show()

In [ ]:
print(f"Variables con datos faltantes: {len(miss)} de {df.shape[1]}")
miss.round(1).to_frame('% faltante').head(6)

**Lectura.** El objetivo está desbalanceado: en cerca de 3 de cada 4 días no llueve al día siguiente. Además, casi todas las variables tienen datos faltantes. Las peores son `Sunshine`, `Evaporation`, `Cloud3pm` y `Cloud9am`, con entre 38% y 48% de faltantes. En el caso de `Sunshine`, 19 ciudades no tienen ni un solo dato, o sea que no es un faltante al azar: hay estaciones que simplemente no miden esa variable. No hay filas duplicadas, pero sí aparecen 3 registros con nubosidad 9, que no es un valor válido en octas.

### Figura 2 - Distribución de tres variables representativas
Elegimos una variable casi simétrica (temperatura máxima), una muy asimétrica (lluvia) y una con cola a la derecha (ráfaga de viento). La lluvia se recorta al percentil 99 solo para graficarla; los cálculos usan todos los datos.

In [ ]:
tmax = df['MaxTemp'].dropna()
lluvia = df['Rainfall'].dropna()
racha = df['WindGustSpeed'].dropna()
paneles = [
    (tmax, 'Temperatura máxima', '°C', 'media', tmax.mean()),
    (lluvia.clip(upper=lluvia.quantile(0.99)), 'Lluvia diaria (≤P99)', 'mm', 'mediana', lluvia.median()),
    (racha, 'Ráfaga máxima de viento', 'km/h', 'mediana', racha.median()),
]

fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
for ax, (x, titulo, xlab, nombre, ref) in zip(axes, paneles):
    ax.hist(x, bins=40, histtype='stepfilled', color=TEAL, alpha=0.3,
            edgecolor=TEAL, linewidth=1.2)
    ax.axvline(ref, color=OCHRE, ls='--', lw=1.8)
    ax.text(ref, ax.get_ylim()[1] * 0.93, f' {nombre}={ref:,.1f}', color=TERRA, fontsize=9, va='center')
    ax.set_title(titulo)
    ax.set_xlabel(xlab)
    ax.grid(axis='x', visible=False)
axes[0].set_ylabel('Frecuencia')

plt.tight_layout()
plt.savefig(FIGURAS / 'fig_distribuciones.png', bbox_inches='tight')
plt.show()

In [ ]:
# media vs mediana y asimetría de las tres variables
pd.DataFrame({
    'Media': [tmax.mean(), lluvia.mean(), racha.mean()],
    'Mediana': [tmax.median(), lluvia.median(), racha.median()],
    'Asimetría': [tmax.skew(), lluvia.skew(), racha.skew()],
}, index=['MaxTemp', 'Rainfall', 'WindGustSpeed']).round(2)

**Lectura.** La temperatura máxima tiene una forma casi simétrica, con la media y la mediana muy parecidas. La lluvia es muy distinta: la mediana es 0 porque en la mayoría de los días no llueve nada, y los pocos días con lluvia forman una cola muy larga hacia la derecha (hasta 371 mm). Por eso para la lluvia conviene usar la mediana y no la media. La ráfaga de viento se concentra entre 30 y 50 km/h y también tiene algo de cola hacia la derecha, aunque bastante menos que la lluvia.

### Figura 3 - Correlación entre variables numéricas

In [ ]:
from matplotlib.colors import LinearSegmentedColormap

num = df_lim.select_dtypes(include='number').columns
corr = df_lim[num].corr()
k = len(corr)
mask = np.triu(np.ones((k, k), dtype=bool), k=1)
cmap = LinearSegmentedColormap.from_list('div', [TERRA, '#ede4d6', TEAL])

fig, ax = plt.subplots(figsize=(6.4, 5.6))
ax.set_facecolor('white'); ax.grid(False)
im = ax.pcolormesh(np.ma.masked_array(corr.values, mask), cmap=cmap, vmin=-1, vmax=1,
                   edgecolors='white', linewidth=1.2)
ax.invert_yaxis()
for i in range(k):
    for j in range(i + 1):
        v = corr.values[i, j]
        if i != j and abs(v) >= 0.6:
            ax.text(j + 0.5, i + 0.5, f'{v:.2f}', ha='center', va='center', fontsize=6.5,
                    color='white' if abs(v) > 0.8 else '#333333')
ax.set_xticks(np.arange(k) + 0.5); ax.set_xticklabels(corr.columns, rotation=90, fontsize=8)
ax.set_yticks(np.arange(k) + 0.5); ax.set_yticklabels(corr.columns, fontsize=8)
ax.set_title('Correlación de Pearson entre variables numéricas')
cb = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04); cb.set_label('r de Pearson')
plt.tight_layout()
plt.savefig(FIGURAS / 'fig_correlacion.png', bbox_inches='tight')
plt.show()

pares = corr.where(np.tril(np.ones((k, k), dtype=bool), k=-1)).stack()
pares.reindex(pares.abs().sort_values(ascending=False).index).head(12).round(2)

Variables relacionadas:
Temperaturas: MaxTemp–Temp3pm (r = 0,98), MinTemp–Temp9am (0,90), MaxTemp–Temp9am (0,89) y Temp9am–Temp3pm (0,86). 

Presiones: Pressure9am–Pressure3pm (0,96). Humedades: Humidity9am–Humidity3pm (0,67), más moderada. 

Las horas de sol se oponen a la nubosidad (Sunshine–Cloud3pm, r = -0,70) y a la humedad de la tarde (-0,63), y la velocidad de la ráfaga se relaciona con la del viento a las 3 pm (0,69).

## 2. Estimación puntual e intervalos de confianza (ID1.3)

Estimamos parámetros poblacionales con un nivel de confianza del 95%. Igual que en el ejemplo del profesor, usamos el intervalo basado en la aproximación normal (método de Wald), que el Teorema Central del Límite justifica por el gran tamaño de la muestra:

- Media: $\bar{x} \pm z_{0.975}\,\dfrac{s}{\sqrt{n}}$
- Proporción: $\hat{p} \pm z_{0.975}\sqrt{\dfrac{\hat{p}(1-\hat{p})}{n}}$

Elegimos parámetros ligados al problema de decisión: la **proporción de días con lluvia al día siguiente** (el parámetro central) y la **media** de tres variables. Son una casi simétrica (`MaxTemp`), una muy asimétrica (`Rainfall`) y una muy relacionada con la lluvia (`Humidity3pm`). Cada intervalo se calcula con los registros que tienen dato en esa variable.

In [ ]:
from scipy import stats

z = stats.norm.ppf(0.975)   # 1.96

def ic_media(x):
    x = pd.Series(x).dropna()
    m = x.mean()
    se = x.std(ddof=1) / np.sqrt(len(x))
    return m, m - z*se, m + z*se, len(x)

def ic_prop(k, n):
    p = k / n
    se = np.sqrt(p*(1 - p) / n)
    return p, p - z*se, p + z*se

filas = []
for col, nombre in [('MaxTemp', 'Temperatura máxima media (°C)'),
                    ('Rainfall', 'Lluvia diaria media (mm)'),
                    ('Humidity3pm', 'Humedad media a las 3 pm (%)')]:
    m, lo, hi, nn = ic_media(df[col])
    filas.append([nombre, m, lo, hi, nn])

y = df[TARGET].dropna()
p, lo, hi = ic_prop((y == 'Yes').sum(), len(y))
filas.append(['Proporción de días con lluvia al día siguiente', p, lo, hi, len(y)])

tabla2 = pd.DataFrame(filas, columns=['Parámetro', 'Estimación', 'IC95% inf', 'IC95% sup', 'n']).set_index('Parámetro')
tabla2.round(4)

**Tabla 2.** Estimación puntual e intervalos de confianza al 95%.

Para que la estimación sirva a la decisión, estimamos también la proporción de lluvia al día siguiente **según si llovió hoy** (`RainToday`).

In [ ]:
sub = df.dropna(subset=[TARGET, 'RainToday'])
filas = []
for grupo, g in sub.groupby('RainToday'):
    k, nn = (g[TARGET] == 'Yes').sum(), len(g)
    p, lo, hi = ic_prop(k, nn)
    filas.append([f'Hoy llovió = {grupo}', p, lo, hi, nn])

p, lo, hi = ic_prop((y == 'Yes').sum(), len(y))
filas.append(['Global', p, lo, hi, len(y)])

ic_cond = pd.DataFrame(filas, columns=['Grupo', 'Proporción', 'IC95% inf', 'IC95% sup', 'n']).set_index('Grupo')
ic_cond.round(4)

### Figura 4 - Probabilidad de lluvia al día siguiente con su intervalo de confianza

In [ ]:
orden = ['Global', 'Hoy llovió = No', 'Hoy llovió = Yes']
d = ic_cond.loc[orden] * [100, 100, 100, 1]   # proporción y límites en %
colores = [OCHRE, TEAL, TERRA]
pos = np.arange(len(d))

fig, ax = plt.subplots(figsize=(9, 3.6))
ax.hlines(pos, 0, d['Proporción'], color=STEM, lw=2.5)
ax.errorbar(d['Proporción'], pos,
            xerr=[d['Proporción'] - d['IC95% inf'], d['IC95% sup'] - d['Proporción']],
            fmt='none', ecolor='#333333', capsize=6, lw=1.2, zorder=4)
ax.scatter(d['Proporción'], pos, s=80, color=colores, edgecolor='#333333', linewidth=0.6, zorder=5)
for i, (pp, lo, hi) in enumerate(d[['Proporción', 'IC95% inf', 'IC95% sup']].values):
    ax.text(hi + 1.2, i, f'{pp:.1f}%  [{lo:.1f}; {hi:.1f}]', va='center', fontsize=9)
ax.set_yticks(pos)
ax.set_yticklabels(['Global', 'Hoy no llovió', 'Hoy sí llovió'])
ax.invert_yaxis()
ax.set_xlim(0, d['IC95% sup'].max() * 1.45)
ax.set_xlabel('% de días con lluvia al día siguiente')
ax.set_title('Probabilidad estimada de lluvia mañana (IC 95%)')
ax.grid(axis='y', visible=False)

plt.tight_layout()
plt.savefig(FIGURAS / 'fig_intervalos.png', bbox_inches='tight')
plt.show()

**Lectura.** Todos los intervalos son muy estrechos. Es lo esperable con más de 140.000 registros: el error estándar se reduce con $\sqrt{n}$.

- **Lluvia al día siguiente.** Llueve al día siguiente en cerca de 1 de cada 4,5 días: el intervalo de confianza al 95 % es [22,20 %; 22,63 %]. El 95 % describe el procedimiento: en muestreos repetidos, cerca del 95 % de los intervalos construidos así contendría el parámetro; no es la probabilidad de que el parámetro esté dentro de este intervalo. Este valor es la «línea base» contra la cual habrá que comparar cualquier modelo: uno que siempre prediga «no llueve» acierta cerca del 78 % de las veces sin aprender nada.
- **Temperatura máxima.** La media está entre 23,18 y 23,26 °C. Como la variable es casi simétrica, la media representa bien a un día típico.
- **Lluvia diaria.** La media está entre 2,32 y 2,40 mm, pero la mediana es 0 mm. La media no describe un día típico: la empujan unos pocos días de lluvia muy intensa. Aun así, el intervalo es válido. Con este $n$, el Teorema Central del Límite asegura que la *media muestral* se distribuye de forma aproximadamente normal, aunque la variable original no lo sea.
- **Humedad a las 3 pm.** La media está entre 51,43% y 51,65%.

**Vínculo con la decisión.** La Figura 3 muestra que la estimación cambia mucho según lo que pasó hoy:

- Si hoy llovió, la probabilidad de lluvia mañana es **46,4%** (IC 95%: 45,9%–47,0%).
- Si no llovió, baja a **15,2%** (IC 95%: 15,0%–15,4%).

Los intervalos de los dos grupos no se superponen, lo que indica que la diferencia no se debe al azar del muestreo (lo pondremos a prueba formalmente en la sección de pruebas de hipótesis). Para quien decide, por ejemplo si conviene cancelar una actividad al aire libre, saber si hoy llovió triplica la probabilidad estimada de lluvia mañana. Eso es mucho más útil que la tasa global. Los grupos suman 140.787 registros, algo menos que el total, porque 1.406 días tienen dato de `RainTomorrow` pero no de `RainToday`.

**Precaución.** Las fórmulas suponen observaciones independientes. Aquí son días consecutivos en las mismas 49 ciudades, y el clima de un día se parece al del día siguiente. Por eso la información "real" es algo menor que $n$, y los intervalos probablemente son algo más angostos de lo que deberían. No cambia las conclusiones, pero conviene tenerlo presente.

## 3. Pruebas de hipótesis (ID1.4)

Nivel de significancia: $\alpha = 0{,}05$. Hacemos tres pruebas, una por cada combinación de tipos de variable, siempre contra la variable objetivo `RainTomorrow`:

- *t* de Welch: una variable continua (`Humidity3pm`) entre días con y sin lluvia al día siguiente.
- $\chi^2$ de independencia: una variable ordinal agrupada (`Cloud3pm`) y el objetivo.
- *z* de dos proporciones: `RainToday` y el objetivo.

### 3.1 Prueba *t* de Welch: humedad a las 3 pm según si llueve mañana
$H_0:\ \mu_{\text{lluvia}} = \mu_{\text{sin lluvia}}$ vs. $H_1:\ \mu_{\text{lluvia}} \neq \mu_{\text{sin lluvia}}$.
Levene rechaza la igualdad de varianzas (p = 0,0002) y los tamaños de grupo son muy desiguales (30.913 y 107.670), por lo que se decide usar Welch

In [ ]:
g_si = df.loc[df[TARGET] == 'Yes', 'Humidity3pm'].dropna()   # llovió mañana
g_no = df.loc[df[TARGET] == 'No',  'Humidity3pm'].dropna()   # no llovió mañana
print(f"Varianzas: lluvia = {g_si.var():.1f}, sin lluvia = {g_no.var():.1f}")

t, p = stats.ttest_ind(g_si, g_no, equal_var=False)
n1_, n0_ = len(g_si), len(g_no)
sp = np.sqrt(((n1_ - 1) * g_si.var(ddof=1) + (n0_ - 1) * g_no.var(ddof=1)) / (n1_ + n0_ - 2))   # desviación agrupada
d = (g_si.mean() - g_no.mean()) / sp

# IC 95% de la diferencia de medias (Welch)
dif = g_si.mean() - g_no.mean()
ee = np.sqrt(g_si.var(ddof=1) / n1_ + g_no.var(ddof=1) / n0_)

print(f"Humedad 3 pm, días con lluvia mañana : {g_si.mean():.2f}% (n={n1_:,})")
print(f"Humedad 3 pm, días sin lluvia mañana : {g_no.mean():.2f}% (n={n0_:,})")
print(f"Diferencia de medias = {dif:.2f} puntos  (IC 95%: {dif - z*ee:.2f} a {dif + z*ee:.2f})")
print(f"t de Welch = {t:.1f}   valor p {'< 0.0001' if p < 1e-4 else f'= {p:.4f}'}   d de Cohen = {d:.2f}")
print('Decisión:', 'se rechaza H0' if p < 0.05 else 'no se rechaza H0')

### Figura 5 - Humedad a las 3 pm según si llueve al día siguiente

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
bins = np.linspace(0, 100, 21)
ax.hist(g_no, bins=bins, density=True, histtype='stepfilled', color=TEAL, alpha=0.30,
        edgecolor=TEAL, linewidth=1.2, label='No llueve mañana')
ax.hist(g_si, bins=bins, density=True, histtype='stepfilled', color=TERRA, alpha=0.30,
        edgecolor=TERRA, linewidth=1.2, label='Llueve mañana')
ax.axvline(g_no.mean(), color=TEAL, ls='--', lw=1.8)
ax.axvline(g_si.mean(), color=TERRA, ls='--', lw=1.8)
ax.text(g_no.mean() - 1.5, ax.get_ylim()[1] * 0.95, f'media={g_no.mean():.1f}', color=TEAL, fontsize=9, ha='right', va='center')
ax.text(g_si.mean() + 1.5, ax.get_ylim()[1] * 0.95, f'media={g_si.mean():.1f}', color=TERRA, fontsize=9, ha='left', va='center')
ax.set_xlabel('Humedad a las 3 pm (%)')
ax.set_ylabel('Densidad')
ax.set_title('Humedad a las 3 pm según si llueve al día siguiente')
ax.legend(frameon=False, loc='upper left')
ax.grid(axis='x', visible=False)

plt.tight_layout()
plt.savefig(FIGURAS / 'fig_humedad_grupos.png', bbox_inches='tight')
plt.show()

**Lectura.** Se rechaza $H_0$. Los días que preceden a una lluvia tienen una humedad media de 68,8% a las 3 pm, contra 46,5% en los días que preceden a uno sin lluvia: una diferencia de 22,3 puntos (IC 95%: 22,05 a 22,53). El tamaño del efecto es grande ($d = 1{,}20$): las dos distribuciones se separan claramente (Figura 4), aunque se traslapan en la zona intermedia.

### 3.2 Prueba $\chi^2$ de independencia: nubosidad a las 3 pm y lluvia mañana
$H_0$: la nubosidad a las 3 pm y la lluvia del día siguiente son independientes. Agrupamos las octas en tres tramos: despejado (0-2), parcial (3-5) y cubierto (6-8). Se excluyen los registros sin `Cloud3pm` y los 3 con valor 9, que no es válido en octas.

In [ ]:
sub = df.dropna(subset=[TARGET, 'Cloud3pm'])
n_excl = df[TARGET].notna().sum() - len(sub)
sub = sub[sub['Cloud3pm'] <= 8]          # excluye los registros con nubosidad 9 (inválidos)
tramos = pd.cut(sub['Cloud3pm'], bins=[-1, 2, 5, 8], labels=['0-2 (despejado)', '3-5 (parcial)', '6-8 (cubierto)'])
tabla = pd.crosstab(tramos, sub[TARGET])
chi2, p_chi, gl, esperadas = stats.chi2_contingency(tabla)
V = np.sqrt(chi2 / (tabla.values.sum() * (min(tabla.shape) - 1)))

print(f"Registros usados: {tabla.values.sum():,}  (se excluyen {n_excl:,} sin Cloud3pm y los de nubosidad 9)")
print(f"Chi² = {chi2:,.1f}   gl = {gl}   valor p {'< 0.0001' if p_chi < 1e-4 else f'= {p_chi:.4f}'}   V de Cramér = {V:.2f}")
print(f"Frecuencia esperada mínima: {esperadas.min():.0f} (supuesto >= 5 se cumple)")
tasa_nubes = (tabla['Yes'] / tabla.sum(axis=1) * 100)
tasa_nubes.round(1).to_frame('% lluvia mañana')

### Figura 6 - Probabilidad de lluvia mañana según la nubosidad de hoy

In [ ]:
media_global = (df[TARGET].dropna() == 'Yes').mean() * 100
x = np.arange(len(tasa_nubes))

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.vlines(x, 0, tasa_nubes.values, color=STEM, lw=3)
ax.scatter(x, tasa_nubes.values, s=150, color=TEAL, edgecolor='#222222', linewidth=0.6, zorder=3)
for xi, v in zip(x, tasa_nubes.values):
    ax.text(xi, v + 2.2, f'{v:.1f}%', ha='center', fontsize=10)
ax.axhline(media_global, color=OCHRE, ls='--', lw=2, label=f'media global = {media_global:.1f}%')
ax.set_xticks(x)
ax.set_xticklabels(tasa_nubes.index)
ax.set_xlabel('Nubosidad a las 3 pm (octas)')
ax.set_ylabel('% de días con lluvia mañana')
ax.set_xlim(-0.5, len(x) - 0.5)
ax.set_ylim(0, tasa_nubes.max() * 1.22)
ax.set_title('Probabilidad de lluvia mañana según la nubosidad de hoy')
ax.legend(frameon=False, loc='upper left')
ax.grid(axis='x', visible=False)

plt.tight_layout()
plt.savefig(FIGURAS / 'fig_nubosidad.png', bbox_inches='tight')
plt.show()

**Lectura.** Se rechaza $H_0$ ($\chi^2 = 10.812{,}6$; 2 gl; $p < 0{,}0001$). La probabilidad de lluvia mañana sube de 5,7% con cielo despejado a 16,9% con nubosidad parcial y 39,6% con cielo cubierto (Figura 5). El efecto es de moderado a grande ($V = 0{,}36$). Como `Cloud3pm` tiene cerca de 40% de datos faltantes, y no al azar (hay estaciones que no la miden), la conclusión vale para los 85.098 registros con dato.


### 3.3 Prueba *z* de dos proporciones: ¿llovió hoy?
Es la prueba formal de lo que mostró la Figura 3. $H_0:\ p_{\text{hoy llovió}} = p_{\text{hoy no llovió}}$ vs. $H_1:\ p_{\text{hoy llovió}} \neq p_{\text{hoy no llovió}}$, con $z = (\hat p_1-\hat p_0)\big/\sqrt{\hat p(1-\hat p)(1/n_1+1/n_0)}$ y $\hat p$ la proporción combinada.

In [ ]:
dv = df.dropna(subset=[TARGET, 'RainToday'])
g1 = (dv.loc[dv['RainToday'] == 'Yes', TARGET] == 'Yes')    # hoy llovió
g0 = (dv.loc[dv['RainToday'] == 'No',  TARGET] == 'Yes')    # hoy no llovió

p1, p0 = g1.mean(), g0.mean()
n1, n0 = len(g1), len(g0)
p_comb = (g1.sum() + g0.sum()) / (n1 + n0)
z_obs = (p1 - p0) / np.sqrt(p_comb * (1 - p_comb) * (1 / n1 + 1 / n0))
p_z = 2 * stats.norm.sf(abs(z_obs))

print(f"Hoy llovió    : {p1*100:.1f}% llueve mañana  (n={n1:,})")
print(f"Hoy no llovió : {p0*100:.1f}% llueve mañana  (n={n0:,})")
print(f"z = {z_obs:.1f}   valor p {'< 0.0001' if p_z < 1e-4 else f'= {p_z:.4f}'}   riesgo relativo = {p1/p0:.1f}")
print('Decisión:', 'se rechaza H0' if p_z < 0.05 else 'no se rechaza H0')

**Lectura.** Se rechaza $H_0$. Si hoy llovió, la probabilidad de lluvia mañana es 46,4%; si no llovió, 15,2%. Es decir, los días con lluvia hoy se asocian con una probabilidad de lluvia mañana 3,1 veces mayor (riesgo relativo = 3,1; $z = 117{,}5$).

**Precaución.** Tenemos más de 140.000 registros, y con tantos datos casi cualquier diferencia sale "significativa" ($p < 0{,}0001$). Por eso nos fijamos más en qué tan grande es el efecto ($d$, $V$ y el riesgo relativo) que en el valor $p$. Además, los datos son días seguidos de las mismas 49 ciudades, y un día se parece al siguiente, así que los valores $p$ son algo exagerados. Por último, estas pruebas muestran que dos variables van juntas, no que una cause a la otra: la humedad y la nubosidad de hoy sirven para anticipar la lluvia de mañana, pero no la provocan.

## 4. Síntesis

- **Descriptiva:** el objetivo está desbalanceado (22,4% de días con lluvia al día siguiente) y 21 de las 23 variables tienen datos faltantes. Hay faltantes que no son al azar (19 ciudades no miden `Sunshine`), una variable muy asimétrica (`Rainfall`, mediana 0 mm) y 3 valores de nubosidad inválidos. No hay filas duplicadas.
- **Estimación:** los intervalos son muy estrechos por el tamaño de la muestra. La proporción de lluvia mañana es 22,42% (IC 95%: 22,20% a 22,63%), y sube de 15,2% a 46,4% si hoy llovió.
- **Pruebas:** las tres rechazan $H_0$ con efectos grandes: la humedad a las 3 pm difiere entre grupos ($d = 1{,}20$), la nubosidad se asocia con la lluvia ($V = 0{,}36$) y los días con lluvia hoy se asocian con una probabilidad 3,1 veces mayor de lluvia mañana (riesgo relativo = 3,1).
- **Alcance y próximos pasos:** son pruebas de a una variable, con dependencia temporal entre observaciones. Cualquier modelo posterior deberá superar la línea base de ~78% de acierto de predecir siempre "no llueve", y deberá definir cómo tratar los faltantes de `Cloud3pm` y `Sunshine`.